# Motifs journaliers de mobilité

Ce notebook montre comment extraire les motifs journaliers de déplacement à partir des staypoints transformés. Un motif est la séquence canonique des lieux visités en une journée par un même utilisateur.

Fonctions couvertes : `build_mobility_motifs()`, `assign_mobility_motif_ids()`, `summarize_mobility_motifs()`, `plot_mobility_motif_graphs()`, `build_mobility_motif_sequences()`.

## 1. Préparer l'environnement

In [ ]:
import pandas as pd

import xyt_gps as xyt

## 2. Générer les données et transformer

Le générateur produit un jeu de traces synthétiques à partir du sample autorisé du package. Les données sont destinées à la démonstration, pas à l'analyse d'un territoire réel.

In [ ]:
phases = (
    xyt.Phase("Phase1", "2026-04-01", "2026-04-14"),
    xyt.Phase("Phase2", "2026-04-15", "2026-04-28"),
)

config = xyt.ProjectConfig(
    experiment_name="demo-motifs",
    raw_data_dir=".",
    phases=phases,
)

synthetic = xyt.generate_synthetic_declic_gps(
    experiments=(xyt.SyntheticExperiment("demo-motifs", phases=phases),),
    users_per_experiment=12,
    random_state=7,
    validate=True,
)

raw, dataset, indicators = xyt.run_mobility_pipeline(
    config,
    raw=synthetic.raw,
    resample_missing_days=True,
    mode_col="mode_niv1",
)

print(f"legs : {len(dataset.legs):,} lignes")
print(f"staypoints : {len(dataset.staypoints):,} lignes")

## 3. Construire les motifs journaliers

`build_mobility_motifs()` extrait, pour chaque utilisateur et chaque journée, la séquence des lieux visités après suppression des répétitions consécutives. Chaque séquence est encodée sous forme d'une matrice d'adjacence aplatie (champ `motif_signature`) qui permet de regrouper les journées structurellement identiques.

La fonction s'appuie sur `purpose_niv1` et les coordonnées lon/lat des staypoints pour construire des clés de lieux reproductibles.

In [ ]:
motifs = xyt.build_mobility_motifs(dataset)

print(f"{len(motifs):,} journées · {motifs['user_id'].nunique()} utilisateurs · {motifs['motif_signature'].nunique()} motifs distincts")
motifs[["user_id", "date", "n_locations", "n_transitions", "is_round_trip", "motif_signature"]].head()

## 4. Assigner les identifiants de motif

`assign_mobility_motif_ids()` numérote les motifs les plus fréquents de 1 à `top_n` (défaut : 9). Les motifs moins fréquents reçoivent l'id 99. Les identifiants sont stables entre exécutions sur les mêmes données.

In [ ]:
motifs = xyt.assign_mobility_motif_ids(motifs, top_n=9)

motifs["motif_id"].value_counts().sort_index().rename("journées").to_frame()

## 5. Résumer la distribution des motifs

`summarize_mobility_motifs()` produit une table de fréquences avec les propriétés structurelles de chaque motif : nombre moyen de lieux visités, de transitions, et proportion de journées en aller-retour.

In [ ]:
summary = xyt.summarize_mobility_motifs(motifs)

(
    summary
    .style
    .format({
        "days_share": "{:.1%}",
        "mean_locations": "{:.1f}",
        "mean_transitions": "{:.1f}",
        "round_trip_share": "{:.1%}",
    })
    .bar(subset=["days_share"], color="#d9cce9", vmin=0, vmax=1)
    .set_caption("Distribution des motifs journaliers")
)

## 6. Visualiser les motifs sous forme de graphes

`plot_mobility_motif_graphs()` reprend l'idée de l'ancien `GPStoGraph.plot_motif()` : chaque motif fréquent est représenté comme un graphe orienté. Le nœud 0 correspond au premier lieu de la séquence canonique, puis les flèches indiquent les transitions entre lieux.

Le graphe décrit la structure d'une journée, pas les lieux réels : deux utilisateurs peuvent avoir le même motif même si leurs coordonnées sont différentes.

In [ ]:
motif_graphs = xyt.plot_mobility_motif_graphs(
    motifs,
    top_n=9,
    title="Motifs journaliers les plus fréquents",
)

motif_graphs

## 7. Séquences journalières par utilisateur

`build_mobility_motif_sequences()` produit une matrice large : une ligne par utilisateur, une colonne par jour de protocole. La valeur est l'identifiant du motif observé ce jour-là (0 = jour non observé ou sans staypoints).

La matrice permet de visualiser la stabilité ou l'évolution des comportements sur la durée du protocole.

In [ ]:
sequences = xyt.build_mobility_motif_sequences(motifs, n_days=28, align_to_week=True)

day_cols = [c for c in sequences.columns if c.startswith("day_")]

MOTIF_COLORS = {
    0:  "#f0eef4",
    1:  "#6b4c9a",
    2:  "#4a90a4",
    3:  "#f5a623",
    4:  "#e84b8a",
    5:  "#3d8f6f",
    6:  "#b8d44a",
    7:  "#2e6b7e",
    8:  "#c07820",
    9:  "#7e7494",
    99: "#e0dce6",
}

def color_motif(val):
    bg = MOTIF_COLORS.get(int(val), "#f8f7f4")
    fg = "#fff" if val not in (0, 6, 99) else "#7e7494"
    return f"background-color: {bg}; color: {fg}; font-size: 0.75rem; text-align: center;"

(
    sequences
    .set_index("user_id")[day_cols]
    .style
    .applymap(color_motif, subset=day_cols)
    .set_caption("Séquences journalières de motifs par utilisateur (0 = non observé, 99 = motif rare)")
    .set_table_styles([{"selector": "th", "props": [("font-size", "0.7rem"), ("padding", "2px 4px")]}])
)

## À retenir

La chaîne `build_mobility_motifs` → `assign_mobility_motif_ids` → `summarize_mobility_motifs` → `build_mobility_motif_sequences` produit quatre niveaux de lecture :

| Table | Grain | Usage principal |
|---|---|---|
| `motifs` (long) | journée × utilisateur | base de travail, jointure avec legs |
| `summary` | motif | fréquences, structure, aller-retour |
| `sequences` (large) | utilisateur × jour | visualisation temporelle |

Le champ `motif_signature` est portable : deux cohortes produites indépendamment peuvent être comparées en utilisant la signature comme clé commune, sans qu'une numérotation commune soit nécessaire.